# JurisFind â€” Kaggle GPU Embedder

**Step 2b of the Qdrant Ingestion Pipeline**

This notebook:
1. Installs `sentence-transformers`
2. Loads `chunks_for_embedding_*.jsonl` from `/kaggle/input/`
3. Embeds all chunks with `all-mpnet-base-v2` on T4 GPU (batch=256)
4. Saves `embeddings.npy` + `chunk_ids.json`

**Upload Instructions:**
- Upload your `.jsonl` file(s) as a Kaggle Dataset
- Add it to this notebook via: Notebook â†’ Data â†’ Add Dataset
- The files will appear at `/kaggle/input/<dataset-name>/`

**Hardware:** GPU T4 x2 | Accelerator: GPU

In [ ]:
# Cell 1 â€” Install dependencies
!pip install -q sentence-transformers==4.1.0

In [ ]:
# Cell 2 â€” Imports and GPU check
import json
import os
import glob
import numpy as np
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
print(f'GPU count       : {torch.cuda.device_count()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name} â€” {props.total_memory / 1024**3:.1f} GB')

In [ ]:
# Cell 3 â€” Find and load input JSONL files
# The dataset files will be at /kaggle/input/<your-dataset-name>/
# Adjust the glob pattern to match your uploaded filenames.

INPUT_GLOB = '/kaggle/input/**/*.jsonl'

jsonl_files = sorted(glob.glob(INPUT_GLOB, recursive=True))
print(f'Found {len(jsonl_files)} JSONL file(s):')
for f in jsonl_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {f}  ({size_mb:.1f} MB)')

if not jsonl_files:
    raise FileNotFoundError(
        'No .jsonl files found! Upload your chunks_for_embedding_*.jsonl '
        'as a Kaggle Dataset and add it to this notebook.'
    )

In [ ]:
# Cell 4 â€” Load all chunks into memory
chunks = []
for jsonl_file in jsonl_files:
    print(f'Loading {jsonl_file} ...')
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                chunks.append(json.loads(line))

print(f'\nTotal chunks loaded: {len(chunks):,}')

texts     = [c['chunk_text'] for c in chunks]
chunk_ids = [c['chunk_id']   for c in chunks]

# Sanity check
assert len(texts) == len(chunk_ids), 'Length mismatch between texts and chunk_ids!'
print(f'Sample text (first 200 chars): {texts[0][:200]}')
print(f'Sample chunk_id: {chunk_ids[0]}')

In [ ]:
# Cell 5 â€” Load model and embed
MODEL_NAME  = 'sentence-transformers/all-mpnet-base-v2'
BATCH_SIZE  = 256   # T4 can handle 256 comfortably
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading model: {MODEL_NAME}')
print(f'Device: {DEVICE}')

model = SentenceTransformer(MODEL_NAME)
model = model.to(DEVICE)

print(f'\nMax sequence length: {model.max_seq_length}')
print(f'Embedding dimension : {model.get_sentence_embedding_dimension()}')
print(f'\nEmbedding {len(texts):,} chunks in batches of {BATCH_SIZE} ...')

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # unit-normalized for cosine similarity
    device=DEVICE,
)

print(f'\nEmbedding complete!')
print(f'Shape: {embeddings.shape}')      # should be (N, 768)
print(f'Dtype: {embeddings.dtype}')      # float32
print(f'Min  : {embeddings.min():.4f}')
print(f'Max  : {embeddings.max():.4f}')
print(f'Norm (first vector): {np.linalg.norm(embeddings[0]):.4f}')  # should be ~1.0

In [ ]:
# Cell 6 â€” Save outputs
OUTPUT_DIR = Path('/kaggle/working')

EMB_PATH  = OUTPUT_DIR / 'embeddings.npy'
IDS_PATH  = OUTPUT_DIR / 'chunk_ids.json'

np.save(str(EMB_PATH), embeddings)
with open(IDS_PATH, 'w', encoding='utf-8') as f:
    json.dump(chunk_ids, f)

emb_size_mb = EMB_PATH.stat().st_size / 1024 / 1024
ids_size_mb = IDS_PATH.stat().st_size / 1024 / 1024

print(f'Saved embeddings.npy  â†’ {EMB_PATH}  ({emb_size_mb:.1f} MB)')
print(f'Saved chunk_ids.json  â†’ {IDS_PATH}  ({ids_size_mb:.1f} MB)')
print(f'\nDownload both files from the Kaggle Output tab.')
print(f'Then run: python scripts/qdrant_ingestion/qdrant_ingestor.py')

In [ ]:
# Cell 7 â€” Optional: verify a small batch of embeddings are normalized
sample_norms = np.linalg.norm(embeddings[:100], axis=1)
print(f'Norms (first 100 vectors):')
print(f'  Min  : {sample_norms.min():.6f}')
print(f'  Max  : {sample_norms.max():.6f}')
print(f'  Mean : {sample_norms.mean():.6f}')
print()
print('All norms should be ~1.0 (cosine similarity ready) âœ“' 
      if sample_norms.min() > 0.99 
      else 'WARNING: norms are not 1.0 â€” check normalize_embeddings=True')